# Stage 2: Data Exploration & Quality Report
This notebook fulfills the practical deliverable for Stage 2: identifying missing values, duplicates, unmatched chargebacks, and transaction counts by date using SQL and pandas.

## 1. Connecting to Database & Loading Data

In [6]:
import pandas as pd
import sqlite3

# Connect to the destination database
conn = sqlite3.connect('destination_db.sqlite')

# Load tables into pandas DataFrames
df_transactions = pd.read_sql_query("SELECT * FROM imported_transactions", conn)
df_customers = pd.read_sql_query("SELECT * FROM imported_customers", conn)
df_chargebacks = pd.read_sql_query("SELECT * FROM imported_chargebacks", conn)

print("Transactions shape:", df_transactions.shape)
print("Customers shape:", df_customers.shape)
print("Chargebacks shape:", df_chargebacks.shape)


Transactions shape: (8, 5)
Customers shape: (12, 5)
Chargebacks shape: (7, 4)


## 2. Data Quality Checks
Checking for missing values, duplicates, and unmatched chargebacks using pandas.

In [7]:
# 1. Missing Values
print("--- Missing Values ---")
print("Transactions:\n", df_transactions.isnull().sum())
print("\nCustomers:\n", df_customers.isnull().sum())
print("\nChargebacks:\n", df_chargebacks.isnull().sum())

# 2. Duplicates (ignoring the primary key 'id' which is always unique)
print("\n--- Duplicate Rows ---")
print("Transactions duplicates:", df_transactions.drop(columns=['id']).duplicated().sum())
print("Customers duplicates:", df_customers.drop(columns=['id']).duplicated().sum())
print("Chargebacks duplicates:", df_chargebacks.drop(columns=['id']).duplicated().sum())

# 3. Unmatched Chargebacks
# A chargeback is unmatched if its transaction_id doesn't exist in the transactions table
unmatched = df_chargebacks[~df_chargebacks['transaction_id'].isin(df_transactions['id'])]
print("\n--- Unmatched Chargebacks ---")
print(unmatched if not unmatched.empty else "No unmatched chargebacks found.")


--- Missing Values ---
Transactions:
 id             0
customer_id    0
amount         2
date           0
status         2
dtype: int64

Customers:
 id             0
name           0
email          2
signup_date    0
risk_score     0
dtype: int64

Chargebacks:
 id                 0
transaction_id     0
chargeback_date    0
reason             0
dtype: int64

--- Duplicate Rows ---
Transactions duplicates: 2
Customers duplicates: 3
Chargebacks duplicates: 3

--- Unmatched Chargebacks ---
   id  transaction_id chargeback_date               reason
4   5            9999      2023-10-11  Unrecognized charge
6   7            9999      2023-10-11  Unrecognized charge


## 3. SQL Exploration: Aggregations, Joins, and Window Functions
We use `pd.read_sql_query` to run complex SQL directly and load the result into pandas.

In [8]:
# Aggregation: Transaction counts by date
query_counts = '''
    SELECT date, COUNT(id) as transaction_count, SUM(amount) as total_amount
    FROM imported_transactions
    GROUP BY date
    ORDER BY date
'''
df_counts = pd.read_sql_query(query_counts, conn)
display(df_counts)


,date,transaction_count,total_amount
0,2023-10-01,1,250.50
1,2023-10-02,2,1215.00
2,2023-10-03,1,75.25
3,2023-10-04,2,1001.50
4,2023-10-10,2,NaN


In [9]:
# Joins & Window Functions: Rank customers by their transaction amounts
query_window = '''
    SELECT 
        c.name,
        t.date,
        t.amount,
        RANK() OVER(PARTITION BY c.id ORDER BY t.amount DESC) as amount_rank
    FROM imported_transactions t
    JOIN imported_customers c ON t.customer_id = c.id
'''
df_window = pd.read_sql_query(query_window, conn)
display(df_window)


,name,date,amount,amount_rank
0,Alice Smith,2023-10-02,1200.00,1
1,Alice Smith,2023-10-01,250.50,2
2,Alice Smith,2023-10-10,NaN,3
3,Alice Smith,2023-10-10,NaN,3
4,Bob Jones,2023-10-02,15.00,1
5,Charlie Brown,2023-10-03,75.25,1
6,Diana Prince,2023-10-04,500.75,1
7,Diana Prince,2023-10-04,500.75,1


In [ ]:
# Close connection when done
conn.close()
